# KinDER Planning Lab

This lab is built on the [KinDER](https://github.com/Princeton-Robot-Planning-and-Learning/kindergarden) benchmark — see the [KinDER website](https://prpl-group.com/kinder-site/) for an overview. We start by defining the environment we will plan in.

## Setup

Run the cell below once to install `kindergarden` and the lightweight 2D-only dependency set (no PyBullet, MuJoCo, pygame, or pymunk).

In [ ]:
# Install kindergarden (kinematic2d, no-PyBullet) plus the bilevel_planning planner.
%pip install -q --no-deps kindergarden
%pip install -q --no-warn-conflicts -r https://raw.githubusercontent.com/Princeton-Robot-Planning-and-Learning/kindergarden/main/requirements/kinematic2d.txt
%pip install -q --no-warn-conflicts bilevel_planning

## The environment

Everything that defines our environment is right here in the notebook so you can read it top to bottom. It is a `kinematic2d` environment: a robot with a movable circular base and a retractable vacuum arm must place a target block onto a target surface that may be obstructed.

### Imports

In [ ]:
"""Obstruction 2D env."""

import inspect
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from relational_structs import Object, ObjectCentricState, Type
from relational_structs.utils import create_state_from_dict

from kinder.core import ConstantObjectKinDEREnv, FinalConfigMeta
from kinder.envs.kinematic2d.base_env import (
    Kinematic2DRobotEnvConfig,
    ObjectCentricKinematic2DRobotEnv,
)
from kinder.envs.kinematic2d.object_types import (
    CRVRobotType,
    Kinematic2DRobotEnvTypeFeatures,
    RectangleType,
)
from kinder.envs.kinematic2d.structs import MultiBody2D, SE2Pose, ZOrder
from kinder.envs.kinematic2d.utils import (
    CRVRobotActionSpace,
    create_walls_from_world_boundaries,
    is_inside,
    is_on,
)
from kinder.envs.utils import PURPLE, sample_se2_pose, state_2d_has_collision

### Object types

The target block and target surface are rectangles with their own types.

In [ ]:
TargetBlockType = Type("target_block", parent=RectangleType)
TargetSurfaceType = Type("target_surface", parent=RectangleType)
Kinematic2DRobotEnvTypeFeatures[TargetBlockType] = list(
    Kinematic2DRobotEnvTypeFeatures[RectangleType]
)
Kinematic2DRobotEnvTypeFeatures[TargetSurfaceType] = list(
    Kinematic2DRobotEnvTypeFeatures[RectangleType]
)

### Configuration

All tunable hyperparameters for the world, robot, table, target, and obstructions.

In [ ]:
@dataclass(frozen=True)
class Obstruction2DEnvConfig(Kinematic2DRobotEnvConfig, metaclass=FinalConfigMeta):
    """Config for Obstruction2DEnv()."""

    # World boundaries. Standard coordinate frame with (0, 0) in bottom left.
    world_min_x: float = 0.0
    world_max_x: float = (1 + np.sqrt(5)) / 2  # golden ratio :)
    world_min_y: float = 0.0
    world_max_y: float = 1.0

    # Action space parameters.
    min_dx: float = -5e-2
    max_dx: float = 5e-2
    min_dy: float = -5e-2
    max_dy: float = 5e-2
    min_dtheta: float = -np.pi / 16
    max_dtheta: float = np.pi / 16
    min_darm: float = -1e-1
    max_darm: float = 1e-1
    min_vac: float = 0.0
    max_vac: float = 1.0

    # Robot hyperparameters.
    robot_base_radius: float = 0.1
    robot_arm_length: float = 2 * robot_base_radius
    robot_gripper_height: float = 0.07
    robot_gripper_width: float = 0.01
    robot_init_pose_bounds: tuple[SE2Pose, SE2Pose] = (
        SE2Pose(
            world_min_x + robot_base_radius,
            world_max_y - 2 * robot_base_radius,
            -np.pi / 2,
        ),
        SE2Pose(
            world_max_x - robot_base_radius, world_max_y - robot_base_radius, -np.pi / 2
        ),
    )

    # Table hyperparameters.
    table_rgb: tuple[float, float, float] = (0.75, 0.75, 0.75)
    table_height: float = 0.1
    table_width: float = world_max_x - world_min_x
    # The table pose is defined relative to the bottom left hand corner.
    table_pose: SE2Pose = SE2Pose(world_min_x, world_min_y, 0.0)

    # Target surface hyperparameters.
    target_surface_rgb: tuple[float, float, float] = PURPLE
    target_surface_init_pose_bounds: tuple[SE2Pose, SE2Pose] = (
        SE2Pose(world_min_x + robot_base_radius, table_pose.y, 0.0),
        SE2Pose(world_max_x - robot_base_radius, table_pose.y, 0.0),
    )
    target_surface_height: float = table_height
    # This adds to the width of the target block.
    target_surface_width_addition_bounds: tuple[float, float] = (
        robot_base_radius / 5,
        robot_base_radius / 2,
    )

    # Target block hyperparameters.
    target_block_rgb: tuple[float, float, float] = PURPLE
    target_block_init_pose_bounds: tuple[SE2Pose, SE2Pose] = (
        SE2Pose(
            world_min_x + robot_base_radius, table_pose.y + table_height + 1e-6, 0.0
        ),
        SE2Pose(
            world_max_x - robot_base_radius, table_pose.y + table_height + 1e-6, 0.0
        ),
    )
    target_block_height_bounds: tuple[float, float] = (
        robot_base_radius / 2,
        2 * robot_base_radius,
    )
    target_block_width_bounds: tuple[float, float] = (
        robot_base_radius / 2,
        2 * robot_base_radius,
    )

    # Obstruction hyperparameters.
    obstruction_rgb: tuple[float, float, float] = (0.75, 0.1, 0.1)
    obstruction_init_pose_bounds = (
        SE2Pose(
            world_min_x + robot_base_radius, table_pose.y + table_height + 1e-6, 0.0
        ),
        SE2Pose(
            world_max_x - robot_base_radius, table_pose.y + table_height + 1e-6, 0.0
        ),
    )
    obstruction_height_bounds: tuple[float, float] = (
        robot_base_radius / 2,
        2 * robot_base_radius,
    )
    obstruction_width_bounds: tuple[float, float] = (
        robot_base_radius / 2,
        2 * robot_base_radius,
    )
    # NOTE: this is not the "real" probability, but rather, the probability
    # that we will attempt to sample the obstruction somewhere on the target
    # surface during each round of rejection sampling during reset().
    obstruction_init_on_target_prob: float = 0.9

    # For sampling initial states.
    max_initial_state_sampling_attempts: int = 10_000

    # For rendering.
    render_dpi: int = 250

### Object-centric environment

The core environment logic: sampling initial states, building states, and the reward/termination rule.

In [ ]:
class ObjectCentricObstruction2DEnv(
    ObjectCentricKinematic2DRobotEnv[Obstruction2DEnvConfig]
):
    """Environment where a block must be placed on an obstructed target."""

    def __init__(
        self,
        num_obstructions: int = 2,
        config: Obstruction2DEnvConfig = Obstruction2DEnvConfig(),
        **kwargs,
    ) -> None:
        super().__init__(config, **kwargs)
        self._num_obstructions = num_obstructions

    def _sample_initial_state(self) -> ObjectCentricState:
        static_objects = set(self.initial_constant_state)
        assert not state_2d_has_collision(
            self.initial_constant_state,
            static_objects,
            static_objects,
            {},
        )
        n = self.config.max_initial_state_sampling_attempts
        for _ in range(n):
            # Sample all randomized values.
            robot_pose = sample_se2_pose(
                self.config.robot_init_pose_bounds, self.np_random
            )
            target_block_pose = sample_se2_pose(
                self.config.target_block_init_pose_bounds, self.np_random
            )
            target_block_shape = (
                self.np_random.uniform(*self.config.target_block_width_bounds),
                self.np_random.uniform(*self.config.target_block_height_bounds),
            )
            target_surface_pose = sample_se2_pose(
                self.config.target_surface_init_pose_bounds, self.np_random
            )
            target_surface_width_addition = self.np_random.uniform(
                *self.config.target_surface_width_addition_bounds
            )
            target_surface_shape = (
                target_block_shape[0] + target_surface_width_addition,
                self.config.target_surface_height,
            )
            obstructions: list[tuple[SE2Pose, tuple[float, float]]] = []
            for _ in range(self._num_obstructions):
                obstruction_shape = (
                    self.np_random.uniform(*self.config.obstruction_width_bounds),
                    self.np_random.uniform(*self.config.obstruction_height_bounds),
                )
                obstruction_init_on_target = (
                    self.np_random.uniform()
                    < self.config.obstruction_init_on_target_prob
                )
                if obstruction_init_on_target:
                    old_lb, old_ub = self.config.obstruction_init_pose_bounds
                    new_x_lb = target_surface_pose.x - obstruction_shape[0]
                    new_x_ub = target_surface_pose.x + target_surface_shape[0]
                    new_lb = SE2Pose(new_x_lb, old_lb.y, old_lb.theta)
                    new_ub = SE2Pose(new_x_ub, old_ub.y, old_ub.theta)
                    pose_bounds = (new_lb, new_ub)
                else:
                    pose_bounds = self.config.obstruction_init_pose_bounds
                obstruction_pose = sample_se2_pose(pose_bounds, self.np_random)
                obstructions.append((obstruction_pose, obstruction_shape))
            state = self._create_initial_state(
                robot_pose,
                target_surface_pose,
                target_surface_shape,
                target_block_pose,
                target_block_shape,
                obstructions,
            )
            # Check initial state validity: goal not satisfied and no collisions.
            if self._target_satisfied(state, {}):
                continue
            full_state = state.copy()
            full_state.data.update(self.initial_constant_state.data)
            all_objects = set(full_state)
            if state_2d_has_collision(full_state, all_objects, all_objects, {}):
                continue
            if self._surface_outside_table(full_state, {}):
                continue
            return state
        raise RuntimeError(f"Failed to sample initial state after {n} attempts")

    def _create_constant_initial_state_dict(self) -> dict[Object, dict[str, float]]:
        init_state_dict: dict[Object, dict[str, float]] = {}

        # Create the table.
        table = Object("table", RectangleType)
        init_state_dict[table] = {
            "x": self.config.table_pose.x,
            "y": self.config.table_pose.y,
            "theta": self.config.table_pose.theta,
            "width": self.config.table_width,
            "height": self.config.table_height,
            "static": True,
            "color_r": self.config.table_rgb[0],
            "color_g": self.config.table_rgb[1],
            "color_b": self.config.table_rgb[2],
            "z_order": ZOrder.ALL.value,
        }

        # Create room walls.
        assert isinstance(self.action_space, CRVRobotActionSpace)
        min_dx, min_dy = self.action_space.low[:2]
        max_dx, max_dy = self.action_space.high[:2]
        wall_state_dict = create_walls_from_world_boundaries(
            self.config.world_min_x,
            self.config.world_max_x,
            self.config.world_min_y,
            self.config.world_max_y,
            min_dx,
            max_dx,
            min_dy,
            max_dy,
        )
        init_state_dict.update(wall_state_dict)

        return init_state_dict

    def _create_initial_state(
        self,
        robot_pose: SE2Pose,
        target_surface_pose: SE2Pose,
        target_surface_shape: tuple[float, float],
        target_block_pose: SE2Pose,
        target_block_shape: tuple[float, float],
        obstructions: list[tuple[SE2Pose, tuple[float, float]]],
    ) -> ObjectCentricState:
        # Shallow copy should be okay because the constant objects should not
        # ever change in this method.
        init_state_dict: dict[Object, dict[str, float]] = {}

        # Create the robot.
        robot = Object("robot", CRVRobotType)
        init_state_dict[robot] = {
            "x": robot_pose.x,
            "y": robot_pose.y,
            "theta": robot_pose.theta,
            "base_radius": self.config.robot_base_radius,
            "arm_joint": self.config.robot_base_radius,  # arm is fully retracted
            "arm_length": self.config.robot_arm_length,
            "vacuum": 0.0,  # vacuum is off
            "gripper_height": self.config.robot_gripper_height,
            "gripper_width": self.config.robot_gripper_width,
        }

        # Create the target surface.
        target_surface = Object("target_surface", TargetSurfaceType)
        init_state_dict[target_surface] = {
            "x": target_surface_pose.x,
            "y": target_surface_pose.y,
            "theta": target_surface_pose.theta,
            "width": target_surface_shape[0],
            "height": target_surface_shape[1],
            "static": True,
            "color_r": self.config.target_surface_rgb[0],
            "color_g": self.config.target_surface_rgb[1],
            "color_b": self.config.target_surface_rgb[2],
            "z_order": ZOrder.NONE.value,
        }

        # Create the target block.
        target_block = Object("target_block", TargetBlockType)
        init_state_dict[target_block] = {
            "x": target_block_pose.x,
            "y": target_block_pose.y,
            "theta": target_block_pose.theta,
            "width": target_block_shape[0],
            "height": target_block_shape[1],
            "static": False,
            "color_r": self.config.target_block_rgb[0],
            "color_g": self.config.target_block_rgb[1],
            "color_b": self.config.target_block_rgb[2],
            "z_order": ZOrder.ALL.value,
        }

        # Create obstructions.
        for i, (obstruction_pose, obstruction_shape) in enumerate(obstructions):
            obstruction = Object(f"obstruction{i}", RectangleType)
            init_state_dict[obstruction] = {
                "x": obstruction_pose.x,
                "y": obstruction_pose.y,
                "theta": obstruction_pose.theta,
                "width": obstruction_shape[0],
                "height": obstruction_shape[1],
                "static": False,
                "color_r": self.config.obstruction_rgb[0],
                "color_g": self.config.obstruction_rgb[1],
                "color_b": self.config.obstruction_rgb[2],
                "z_order": ZOrder.ALL.value,
            }

        # Finalize state.
        return create_state_from_dict(init_state_dict, Kinematic2DRobotEnvTypeFeatures)

    def _target_satisfied(
        self,
        state: ObjectCentricState,
        static_object_body_cache: dict[Object, MultiBody2D],
    ) -> bool:
        target_objects = state.get_objects(TargetBlockType)
        assert len(target_objects) == 1
        target_object = target_objects[0]
        target_surfaces = state.get_objects(TargetSurfaceType)
        assert len(target_surfaces) == 1
        target_surface = target_surfaces[0]
        return is_on(state, target_object, target_surface, static_object_body_cache)

    def _surface_outside_table(
        self,
        state: ObjectCentricState,
        static_object_body_cache: dict[Object, MultiBody2D],
    ) -> bool:
        """Check if the target surface is outside the table boundaries."""
        target_surfaces = state.get_objects(TargetSurfaceType)
        assert len(target_surfaces) == 1
        target_surface = target_surfaces[0]
        table = state.get_objects(RectangleType)
        table = [obj for obj in table if obj.name == "table"]
        assert len(table) == 1
        table_instance = table[0]

        is_inside_table = is_inside(
            state,
            target_surface,
            table_instance,
            static_object_body_cache,
        )
        return not is_inside_table

    def _get_reward_and_done(self) -> tuple[float, bool]:
        # Terminate when target object is on the target surface. Give -1 reward
        # at every step until then to encourage fast completion.
        assert self._current_state is not None
        terminated = self._target_satisfied(
            self._current_state,
            self._static_object_body_cache,
        )
        return -1.0, terminated

## Creating the environment and inspecting its state

An **object-centric state** is a set of objects, each with a *type* and a fixed list of *named features* (e.g. a `crv_robot` has `x`, `y`, `theta`, `arm_joint`, ...). Below we create the environment, render one initial state, and then read that same state programmatically so you can connect the picture to the underlying numbers.

Key API: `state.pretty_str()` prints the whole state as a table, `state.get_objects(SomeType)` finds objects of a type, and `state.get(obj, feature_name)` reads a single feature value.

In [ ]:
# Create the environment: object-centric, with 2 obstructions.
env = ObjectCentricObstruction2DEnv(num_obstructions=1)

# An object-centric state is a set of objects, each with a type and a fixed
# list of named features. Let's look at one concrete initial state.
state, _ = env.reset(seed=0)

# 1. The whole state as a human-readable table (one block per object type).
print(state.pretty_str())

# 2. Render the same state so we can connect the numbers to the picture.
plt.figure(figsize=(6, 4))
plt.imshow(env.render())
plt.axis("off")
plt.title("Obstruction2D initial state (seed=0)")
plt.show()

# 3. Walk the state programmatically: every object knows its type, and we read
#    a feature value with state.get(obj, feature_name).
for obj in sorted(state, key=lambda o: o.name):
    features = Kinematic2DRobotEnvTypeFeatures[obj.type]
    values = {f: round(float(state.get(obj, f)), 3) for f in features}
    print(f"{obj.name} ({obj.type.name}): {values}")

# 4. Query objects by type, and read the robot's pose straight from the state.
robot = state.get_objects(CRVRobotType)[0]
print(
    f"\nrobot pose -> x={state.get(robot, 'x'):.3f}, "
    f"y={state.get(robot, 'y'):.3f}, theta={state.get(robot, 'theta'):.3f}"
)

## Exercise: clear the obstructions ✏️

So far we have only *read* states. Now you will *write* one.

To modify an environment's state directly, build it with `allow_state_access=True`. Then `env.get_state()` returns the current state and `env.set_state(new_state)` overwrites it. The recommended pattern is to copy the current state, edit the copy with `state.set(obj, feature, value)`, and write it back.

**Your task:** starting from the `seed=0` initial state, move every obstruction so that it no longer overlaps the target surface's horizontal span, leaving the target region clear. Fill in the marked section below; the assertion tells you whether you succeeded.

In [ ]:
# Build the environment with state access enabled so we can overwrite its state.
env = ObjectCentricObstruction2DEnv(num_obstructions=1, allow_state_access=True)
state, _ = env.reset(seed=0)

# The target surface occupies the horizontal span [x, x + width]. An obstruction
# blocks the target region whenever its own span overlaps that interval.
target_surface = state.get_objects(TargetSurfaceType)[0]


def overlaps_target(s, obj):
    sx, sw = s.get(target_surface, "x"), s.get(target_surface, "width")
    ox, ow = s.get(obj, "x"), s.get(obj, "width")
    return ox < sx + sw and ox + ow > sx


# Edit a copy of the state, then write it back into the environment.
new_state = state.copy()
# YOUR CODE HERE
env.set_state(new_state)

# Check your work: no obstruction should overlap the target region anymore.
cleared = env.get_state()
still_blocking = [
    o.name
    for o in cleared
    if o.name.startswith("obstruction") and overlaps_target(cleared, o)
]
assert not still_blocking, f"Still blocking the target region: {still_blocking}"
print("Target region is clear!")

plt.figure(figsize=(6, 4))
plt.imshow(env.render())
plt.axis("off")
plt.title("After clearing the obstructions")
plt.show()

## From states to symbols: predicates and classifiers

Recall the two pieces that turn a continuous state into a symbolic one:

- A **predicate** is a named, typed relation — a template for a property of objects, written like `OnTarget(block)` or `Holding(robot, block)`. The bracketed types are its *signature*.
- A predicate's **classifier** is the rule that decides, for a concrete continuous state and specific objects, whether that relation currently holds. Geometry in, a boolean out.

Applying every classifier to a state yields the set of true **ground atoms** (a predicate filled in with concrete objects, e.g. `OnTarget(target_block)`). That set is the **abstract state**, and the function that runs all the classifiers and collects the true atoms is the **state abstractor** — what we build here. Note that we keep every classifier inside one `state_abstractor` function rather than attaching one to each predicate.

In [ ]:
from kinder.envs.kinematic2d.utils import get_suctioned_objects
from relational_structs import GroundAtom, Object, Predicate

# Each Predicate is a name plus the list of argument types (its signature).
Holding = Predicate("Holding", [CRVRobotType, RectangleType])  # robot grasps a block
HandEmpty = Predicate("HandEmpty", [CRVRobotType])             # robot grasps nothing
OnTable = Predicate("OnTable", [RectangleType])                # block rests on the table
OnTarget = Predicate("OnTarget", [RectangleType])              # block rests on the target

predicates = {Holding, HandEmpty, OnTable, OnTarget}
for predicate in sorted(predicates, key=str):
    print(predicate)

### The state abstractor (your turn ✏️)

Below is `state_abstractor`. It already classifies `Holding`, `HandEmpty`, and `OnTarget`. **Your task: complete the `OnTable` branch.**

`OnTable(block)` should be true exactly when the block rests on the table — that is, when it is *neither* being held *nor* on the target surface. You have everything you need:

- `suctioned` is the set of objects the gripper is currently holding.
- `is_on(state, block, target_surface, {})` returns `True` when `block` rests on the surface (the `{}` is an empty geometry cache).
- Add an atom with `atoms.add(GroundAtom(OnTable, [block]))`.

The `OnTarget` branch just above shows the pattern. Fill in the section marked `# YOUR CODE HERE`, then run the next cell to check your work.

In [ ]:
def state_abstractor(state):
    """Map a continuous state to the set of ground atoms that hold in it.

    Runs each predicate's classifier over the relevant objects and collects the
    atoms that are true. The result -- a set of GroundAtoms -- is the abstract
    state that a symbolic planner reasons about.
    """
    # Pull the objects we care about straight out of the state.
    robot = state.get_objects(CRVRobotType)[0]
    target = state.get_objects(TargetBlockType)[0]
    target_surface = state.get_objects(TargetSurfaceType)[0]
    obstructions = {o for o in state if o.name.startswith("obstruction")}
    movable_blocks = obstructions | {target}

    atoms = set()

    # Holding(robot, block): the block is currently suctioned by the gripper.
    suctioned = {o for o, _ in get_suctioned_objects(state, robot)}
    for block in suctioned & movable_blocks:
        atoms.add(GroundAtom(Holding, [robot, block]))

    # HandEmpty(robot): nothing is suctioned.
    if not suctioned:
        atoms.add(GroundAtom(HandEmpty, [robot]))

    # OnTarget(block): the block geometrically rests on the target surface. is_on
    # is the same helper the environment uses to decide when the goal is reached.
    for block in movable_blocks:
        if is_on(state, block, target_surface, {}):
            atoms.add(GroundAtom(OnTarget, [block]))

    # OnTable(block): the block rests on the table -- neither held nor on target.
    # YOUR CODE HERE

    return atoms

### Try it out

Run the abstractor on a concrete state, and watch the abstract state change when we move a block.

In [ ]:
env = ObjectCentricObstruction2DEnv(num_obstructions=1, allow_state_access=True)
state, _ = env.reset(seed=0)

print("Abstract state of the initial scene:")
for atom in sorted(state_abstractor(state), key=str):
    print("  ", atom)

# Classifiers are just functions of the state, so changing the state changes the
# abstract state. Place the target block on the surface and re-abstract.
surface = state.get_objects(TargetSurfaceType)[0]
block = state.get_objects(TargetBlockType)[0]
placed = state.copy()
placed.set(block, "x", state.get(surface, "x") + (state.get(surface, "width") - state.get(block, "width")) / 2)
placed.set(block, "y", state.get(surface, "y") + state.get(surface, "height"))
print("\nAfter placing the target block on the surface:")
for atom in sorted(state_abstractor(placed), key=str):
    print("  ", atom)

# Once your OnTable branch is correct, these checks pass.
init_atoms = state_abstractor(state)
on_table_blocks = {block} | {o for o in state if o.name.startswith("obstruction")}
assert {GroundAtom(OnTable, [b]) for b in on_table_blocks} <= init_atoms, (
    "Every block should be OnTable in the initial scene."
)
assert GroundAtom(OnTarget, [block]) in state_abstractor(placed), (
    "The placed block should be OnTarget."
)
print("\nNice -- OnTable is working!")

## Planning moves: operators

The state abstractor gives us a symbolic snapshot. To *plan*, we also need symbolic **actions** — ways the abstract state can change. An **operator** is a lifted (variable-ized) action model with three parts:

- **parameters** — typed variables like `?robot` and `?block`, so one operator stands for every concrete instance it could apply to;
- **preconditions** — atoms that must hold in the current abstract state for the operator to apply;
- **effects** — *add effects* (atoms that become true) and *delete effects* (atoms that become false).

A planner searches for a sequence of operators whose preconditions and effects chain from the initial abstract state to one that satisfies the goal. We define four: pick a block from the table or from the target, and place a held block onto the table or the target.

In [ ]:
from relational_structs import LiftedAtom, LiftedOperator, Variable

# Variables let one operator cover every robot/block it could apply to.
robot_var = Variable("?robot", CRVRobotType)
block_var = Variable("?block", RectangleType)

PickFromTable = LiftedOperator(
    "PickFromTable",
    [robot_var, block_var],
    preconditions={LiftedAtom(HandEmpty, [robot_var]), LiftedAtom(OnTable, [block_var])},
    add_effects={LiftedAtom(Holding, [robot_var, block_var])},
    delete_effects={LiftedAtom(HandEmpty, [robot_var]), LiftedAtom(OnTable, [block_var])},
)

PickFromTarget = LiftedOperator(
    "PickFromTarget",
    [robot_var, block_var],
    preconditions={LiftedAtom(HandEmpty, [robot_var]), LiftedAtom(OnTarget, [block_var])},
    add_effects={LiftedAtom(Holding, [robot_var, block_var])},
    delete_effects={LiftedAtom(HandEmpty, [robot_var]), LiftedAtom(OnTarget, [block_var])},
)

PlaceOnTable = LiftedOperator(
    "PlaceOnTable",
    [robot_var, block_var],
    preconditions={LiftedAtom(Holding, [robot_var, block_var])},
    add_effects={LiftedAtom(HandEmpty, [robot_var]), LiftedAtom(OnTable, [block_var])},
    delete_effects={LiftedAtom(Holding, [robot_var, block_var])},
)

for operator in [PickFromTable, PickFromTarget, PlaceOnTable]:
    print(operator.pddl_str)

### Define the last operator (your turn ✏️)

Three operators are given above. One is missing: **`PlaceOnTarget`**, which places a held block onto the target surface. It mirrors `PlaceOnTable`, except the block ends up `OnTarget` instead of `OnTable`. It should have:

- **parameters:** `[robot_var, block_var]`
- **precondition:** the robot is `Holding` the block;
- **add effects:** the robot is `HandEmpty`, and the block is `OnTarget`;
- **delete effect:** the robot is no longer `Holding` the block.

Replace `# YOUR CODE HERE` below with the `PlaceOnTarget = LiftedOperator(...)` definition, then run the next cell to plan with it.

In [ ]:
# Define PlaceOnTarget here, modelled on PlaceOnTable (see the spec above).
# YOUR CODE HERE

### Try it out

An operator applies to an abstract state when its preconditions hold; applying it deletes its delete effects and adds its add effects. Let's chain two operators by hand to put the target block on the surface.

In [ ]:
operators = {PickFromTable, PickFromTarget, PlaceOnTable, PlaceOnTarget}


def apply_operator(ground_op, atoms):
    """Apply a ground operator to a set of atoms (its preconditions must hold)."""
    assert ground_op.preconditions <= atoms, f"preconditions not met: {ground_op.short_str}"
    return (atoms - ground_op.delete_effects) | ground_op.add_effects


env = ObjectCentricObstruction2DEnv(num_obstructions=1, allow_state_access=True)
state, _ = env.reset(seed=0)
robot = state.get_objects(CRVRobotType)[0]
target = state.get_objects(TargetBlockType)[0]

# Goal: the target block is on the target surface.
goal = {GroundAtom(OnTarget, [target])}

atoms = state_abstractor(state)
print("start:", sorted(str(a) for a in atoms))

# A two-step plan: pick the target block off the table, then place it on target.
plan = [PickFromTable.ground((robot, target)), PlaceOnTarget.ground((robot, target))]
for ground_op in plan:
    atoms = apply_operator(ground_op, atoms)
    print(ground_op.short_str, "->", sorted(str(a) for a in atoms))

# We reached the goal only at the *symbolic* level: we applied each operator's
# add/delete effects to the set of atoms. The robot has not actually moved --
# turning a symbolic plan into real robot actions is a separate step.
assert goal <= atoms, "Plan did not reach the goal."
print("\nGoal reached -- at the symbolic level!")

## Finding the plan automatically

We hardcoded that two-step plan. But given the same ingredients — the initial abstract state, the operators, and the goal — a **symbolic planner** can *search* for it. We use greedy best-first search from `prpl-utils`: search states are sets of atoms, successors come from applying every operator whose preconditions hold, and a goal-count heuristic guides the search.

In [ ]:
from prpl_utils.search import run_gbfs

# Candidate objects to ground the operators over: the robot and the blocks.
blocks = [target] + [obj for obj in state if obj.name.startswith("obstruction")]


def get_successors(atoms):
    """Yield (operator, next_atoms, cost) for every applicable ground operator."""
    for operator in operators:
        for block in blocks:
            ground_op = operator.ground((robot, block))
            if ground_op.preconditions <= atoms:
                next_atoms = frozenset((atoms - ground_op.delete_effects) | ground_op.add_effects)
                yield ground_op, next_atoms, 1.0


def heuristic(atoms):
    """Number of unsatisfied goal atoms -- zero exactly at a goal state."""
    return len(goal - atoms)


# Search from the initial abstract state for a sequence of operators reaching the goal.
initial_atoms = frozenset(state_abstractor(state))
_states, plan, metrics = run_gbfs(
    initial_atoms,
    check_goal=lambda atoms: goal <= atoms,
    get_successors=get_successors,
    heuristic=heuristic,
)

print(f"Search found a {len(plan)}-step plan ({metrics}):")
for ground_op in plan:
    print("  ", ground_op.short_str)

## From symbols to motion: parameterized options

The symbolic plan said *what* to do — `PickFromTable`, then `PlaceOnTarget` — but not *how* to move the robot. Each operator is paired with a **parameterized option** (a *controller*): given the current state and a few continuous **parameters**, it emits the low-level robot actions that realize the operator.

The parameters capture what the symbolic effects leave open: for a pick, *where along the block to grasp*; for a place, *where to release it*. A controller can `sample_parameters` to propose values, then rolls out an action sequence to carry them out — the bridge from symbolic plans back to continuous control.

The controllers below follow simple waypoints (rise to a safe height, move across, drop down) instead of full motion planning; the environment is simple enough that this always works. Everything is here in the notebook — the only missing piece is one sampler.

### The base controller

`Kinematic2dRobotController` holds the shared machinery: after `reset(state, params)`, calling `step()` repeatedly yields low-level delta actions until `terminated()`. Subclasses only supply the waypoints and the vacuum behavior. `_waypoints_to_plan` splits each waypoint-to-waypoint move into steps no larger than the action space allows.

In [ ]:
from bilevel_planning.structs import GroundParameterizedController


class Kinematic2dRobotController(GroundParameterizedController):
    """Drives the CRV robot along SE2 waypoints to realize an operator."""

    def __init__(self, objects, action_space, safe_y=0.8):
        self._robot = objects[0]
        assert self._robot.is_instance(CRVRobotType)
        super().__init__(objects)
        self._action_space = action_space
        self._current_params = 0.0
        self._current_plan = None
        self._current_state = None
        self._safe_y = safe_y
        self._max_delta_x = action_space.high[0]
        self._max_delta_y = action_space.high[1]
        self._max_delta_theta = action_space.high[2]
        self._max_delta_arm = action_space.high[3]

    def _waypoints_to_plan(self, state, waypoints, vacuum_during_plan):
        """Turn (pose, arm) waypoints into a list of per-step delta actions."""
        start = (
            SE2Pose(
                state.get(self._robot, "x"),
                state.get(self._robot, "y"),
                state.get(self._robot, "theta"),
            ),
            state.get(self._robot, "arm_joint"),
        )
        waypoints = [start] + waypoints
        plan = []
        for a, b in zip(waypoints[:-1], waypoints[1:]):
            if np.allclose(
                [a[0].x, a[0].y, a[0].theta, a[1]],
                [b[0].x, b[0].y, b[0].theta, b[1]],
            ):
                continue
            total_dx = b[0].x - a[0].x
            total_dy = b[0].y - a[0].y
            total_dtheta = b[0].theta - a[0].theta
            if abs(total_dtheta) > np.pi:  # take the shorter way around
                total_dtheta -= 2 * np.pi if total_dtheta > 0 else -2 * np.pi
            total_darm = b[1] - a[1]
            num_steps = int(
                max(
                    np.ceil(abs(total_dx) / self._max_delta_x),
                    np.ceil(abs(total_dy) / self._max_delta_y),
                    np.ceil(abs(total_dtheta) / self._max_delta_theta),
                    np.ceil(abs(total_darm) / self._max_delta_arm),
                )
            )
            action = np.array(
                [
                    total_dx / num_steps,
                    total_dy / num_steps,
                    total_dtheta / num_steps,
                    total_darm / num_steps,
                    vacuum_during_plan,
                ],
                dtype=np.float32,
            )
            plan.extend([action] * num_steps)
        return plan

    def reset(self, state, params):
        """Start a fresh execution with the given parameters."""
        self._current_params = params
        self._current_plan = None
        self._current_state = state

    def terminated(self):
        return self._current_plan is not None and len(self._current_plan) == 0

    def step(self):
        if self._current_plan is None:
            self._current_plan = self._generate_plan(self._current_state)
        return self._current_plan.pop(0)

    def observe(self, state):
        self._current_state = state


def get_robot_transfer_position(block, state, block_x, robot_arm_joint, relative_x_offset=0.0):
    """The (x, y) the robot base needs so its gripper reaches the block."""
    robot = state.get_objects(CRVRobotType)[0]
    surface = state.get_objects(TargetSurfaceType)[0]
    ground = state.get(surface, "y") + state.get(surface, "height")
    padding = 1e-4
    x = block_x + relative_x_offset
    y = (
        ground
        + state.get(block, "height")
        + robot_arm_joint
        + state.get(robot, "gripper_width") / 2
        + padding
    )
    return (x, y)

### Picking

`GroundPickController` grasps a block when the hand is free. Its single parameter is the grasp x *relative to the block*, and its `sample_parameters` proposes a point anywhere across the block (plus a little gripper slack) — a worked example of the kind of sampler you will write below.

In [ ]:
class GroundPickController(Kinematic2dRobotController):
    """Pick up a block. Parameter: grasp x relative to the block center."""

    def __init__(self, objects, action_space):
        super().__init__(objects, action_space)
        self._block = objects[1]
        assert self._block.is_instance(RectangleType)

    def sample_parameters(self, state, rng):
        # Grasp anywhere across the block, with a little gripper slack on each side.
        gripper_height = state.get(self._robot, "gripper_height")
        block_width = state.get(self._block, "width")
        return rng.uniform(-gripper_height / 2, block_width + gripper_height / 2)

    def _generate_waypoints(self, state):
        robot_x = state.get(self._robot, "x")
        robot_theta = state.get(self._robot, "theta")
        arm = state.get(self._robot, "arm_joint")
        block_x = state.get(self._block, "x")
        offset = self._current_params[0] if isinstance(self._current_params, (tuple, list)) else self._current_params
        target_x, target_y = get_robot_transfer_position(self._block, state, block_x, arm, relative_x_offset=offset)
        return [
            (SE2Pose(robot_x, self._safe_y, robot_theta), arm),   # rise to safe height
            (SE2Pose(target_x, self._safe_y, robot_theta), arm),  # move over the block
            (SE2Pose(target_x, target_y, robot_theta), arm),      # drop down to grasp
        ]

    def _get_vacuum_actions(self):
        return 0.0, 1.0  # vacuum off while moving, on to grasp

    def step(self):
        # Fully extend the arm before following the waypoints.
        if self._current_state.get(self._robot, "arm_joint") <= 0.15:
            return np.array([0, 0, 0, self._action_space.high[3], 0], dtype=np.float32)
        return super().step()

    def _generate_plan(self, state):
        waypoints = self._generate_waypoints(state)
        vacuum_during, vacuum_after = self._get_vacuum_actions()
        plan = self._waypoints_to_plan(state, waypoints, vacuum_during)
        plan += [
            np.array([0, 0, 0, 0, vacuum_after], dtype=np.float32),                            # toggle vacuum
            np.array([0, self._action_space.high[1], 0, 0, vacuum_after], dtype=np.float32),   # lift to break contact
        ]
        return plan

### Placing

`_GroundPlaceController` releases a held block at an **absolute** x. `GroundPlaceOnTableController` can drop it anywhere on the table, so its sampler is trivial. Placing on the *target* is more delicate — that is your exercise.

In [ ]:
class _GroundPlaceController(Kinematic2dRobotController):
    """Place a held block. Parameter: absolute x where the block is released."""

    def __init__(self, objects, action_space):
        super().__init__(objects, action_space)
        self._block = objects[1]
        assert self._block.is_instance(RectangleType)

    def _generate_waypoints(self, state):
        robot_x = state.get(self._robot, "x")
        robot_theta = state.get(self._robot, "theta")
        arm = state.get(self._robot, "arm_joint")
        placement_x = self._current_params[0] if isinstance(self._current_params, (tuple, list)) else self._current_params
        target_x, target_y = get_robot_transfer_position(self._block, state, placement_x, arm)
        return [
            (SE2Pose(robot_x, self._safe_y, robot_theta), arm),
            (SE2Pose(target_x, self._safe_y, robot_theta), arm),
            (SE2Pose(target_x, target_y, robot_theta), arm),
        ]

    def _get_vacuum_actions(self):
        return 1.0, 0.0  # hold while moving, release at the end

    def step(self):
        if self._current_state.get(self._robot, "arm_joint") <= 0.15:
            return np.array([0, 0, 0, self._action_space.high[3], 0], dtype=np.float32)
        return super().step()

    def _generate_plan(self, state):
        waypoints = self._generate_waypoints(state)
        vacuum_during, vacuum_after = self._get_vacuum_actions()
        plan = self._waypoints_to_plan(state, waypoints, vacuum_during)
        plan += [
            np.array([0, 0, 0, 0, vacuum_after], dtype=np.float32),
            np.array([0, self._action_space.high[1], 0, 0, vacuum_after], dtype=np.float32),
        ]
        return plan


class GroundPlaceOnTableController(_GroundPlaceController):
    """Place a held block somewhere on the table."""

    def sample_parameters(self, state, rng):
        del state  # any x in the world works for the table
        return rng.uniform(0.0, 1.0)

### Sample where to place on the target (your turn ✏️)

`GroundPlaceOnTargetController` needs its `sample_parameters`. The parameter is the **absolute x where the robot releases the block**, and to land the block on the surface you must account for two things:

1. **The surface extent.** The surface spans `[surface_x, surface_x + surface_width]`, and the block has its own `width`, so valid release positions form a *range*, not a single value.
2. **The grasp offset.** The robot grabbed the block off-center, so the robot's x differs from the block's x by `robot_x - block_x`. The parameter is an x for the *robot*, so add that offset.

Sample uniformly between `surface_x + (robot_x - block_x)` (block's left edge at the surface's left edge) and that plus `surface_width - block_width`. If the two bounds come out reversed — possible when a block is wider than the surface — swap them, then return the sampled value. The `surface` object is fetched for you; use `state.get(obj, feature)` for the rest.

In [ ]:
class GroundPlaceOnTargetController(_GroundPlaceController):
    """Place a held block on the target surface."""

    def sample_parameters(self, state, rng):
        surface = state.get_objects(TargetSurfaceType)[0]
        # YOUR CODE HERE

### Try it out

Now we can realize the symbolic plan `PickFromTable -> PlaceOnTarget` with the controllers: sample each option's parameters and roll it out in the environment. We use a no-obstruction instance so a single pick-and-place reaches the goal; with obstructions you would first run the planner to clear the target region. The rollout plays back as a video you can play, loop, and scrub.

In [ ]:
from matplotlib import animation
from IPython.display import HTML


def execute_option(env, controller, obs, rng, frames):
    """Sample parameters for an option, roll it out, and record a frame per step."""
    params = controller.sample_parameters(obs, rng)
    assert params is not None, "sample_parameters returned None -- is it implemented?"
    controller.reset(obs, params)
    while not controller.terminated():
        obs, _, _, _, _ = env.step(controller.step())
        controller.observe(obs)
        frames.append(env.render())
    return obs


env = ObjectCentricObstruction2DEnv(num_obstructions=0)
obs, _ = env.reset(seed=0)
rng = np.random.default_rng(0)
robot = obs.get_objects(CRVRobotType)[0]
target = obs.get_objects(TargetBlockType)[0]
surface = obs.get_objects(TargetSurfaceType)[0]

# Realize the symbolic plan PickFromTable -> PlaceOnTarget, recording every frame.
frames = [env.render()]
obs = execute_option(env, GroundPickController([robot, target], env.action_space), obs, rng, frames)
obs = execute_option(env, GroundPlaceOnTargetController([robot, target], env.action_space), obs, rng, frames)

assert is_on(obs, target, surface, {}), "The block did not end up on the target."
print(f"The block is on the target -- a fully grounded plan in {len(frames)} steps!")

# Animate the frames into a video player (use the controls to play, loop, or scrub).
fig, ax = plt.subplots(figsize=(6, 4))
ax.axis("off")
image = ax.imshow(frames[0])


def _draw(i):
    image.set_data(frames[i])
    return [image]


anim = animation.FuncAnimation(fig, _draw, frames=len(frames), interval=100, blit=True)
plt.close(fig)  # don't also show the static first frame
HTML(anim.to_jshtml())

## Putting it together: bilevel planning

We now have every ingredient SeSamE needs:

- the **predicates** and **state abstractor** (continuous state -> symbolic atoms),
- the **operators** (symbolic actions),
- the **parameterized controllers** (operators -> low-level actions),
- and the environment itself as a **transition function**.

SeSamE (from the `bilevel_planning` package, installed at the top) searches at the symbolic level for a plan, then refines it by sampling controller parameters until the low-level execution actually works — backtracking to a different symbolic plan if refinement keeps failing. Unlike our earlier hand-built rollout, this can *clear the obstruction* before placing the target, because it searches over alternative symbolic plans.

We pair each operator with its controller to form **skills**, wrap our set-returning abstractor as a `RelationalAbstractState`, add a goal deriver, and hand everything to `SesameModels`.

In [ ]:
from gymnasium.spaces import Box
from relational_structs.spaces import ObjectCentricStateSpace
from bilevel_planning.structs import (
    LiftedParameterizedController,
    LiftedSkill,
    RelationalAbstractGoal,
    RelationalAbstractState,
    SesameModels,
)
from bilevel_planning.sesame import run_sesame

NUM_OBSTRUCTIONS = 1

# A simulator the planner uses to predict the effect of low-level actions, and
# the spaces it reasons over.
sim = ObjectCentricObstruction2DEnv(num_obstructions=NUM_OBSTRUCTIONS)
action_space = sim.action_space
observation_space = sim.observation_space

# Wrap each ground controller as a lifted controller over (?robot, ?block),
# baking in the action space. The parameter space is the [0, 1] interval our
# samplers draw from.
params_space = Box(low=np.array([0.0]), high=np.array([1.0]), dtype=np.float32)


class _PickController(GroundPickController):
    def __init__(self, objects):
        super().__init__(objects, action_space)


class _PlaceOnTableController(GroundPlaceOnTableController):
    def __init__(self, objects):
        super().__init__(objects, action_space)


class _PlaceOnTargetController(GroundPlaceOnTargetController):
    def __init__(self, objects):
        super().__init__(objects, action_space)


pick_controller = LiftedParameterizedController([robot_var, block_var], _PickController, params_space)
place_on_table_controller = LiftedParameterizedController([robot_var, block_var], _PlaceOnTableController, params_space)
place_on_target_controller = LiftedParameterizedController([robot_var, block_var], _PlaceOnTargetController, params_space)

# A skill pairs an operator with the controller that realizes it.
skills = {
    LiftedSkill(PickFromTable, pick_controller),
    LiftedSkill(PickFromTarget, pick_controller),
    LiftedSkill(PlaceOnTable, place_on_table_controller),
    LiftedSkill(PlaceOnTarget, place_on_target_controller),
}


# Our state_abstractor returns a bare set of atoms; SeSamE wants a
# RelationalAbstractState (the atoms plus the objects involved).
def sesame_state_abstractor(state):
    return RelationalAbstractState(state_abstractor(state), set(state))


# The goal is always: the target block is on the target surface.
def goal_deriver(state):
    target = state.get_objects(TargetBlockType)[0]
    return RelationalAbstractGoal({GroundAtom(OnTarget, [target])}, sesame_state_abstractor)


# Predict the next continuous state by simulating the action.
def transition_fn(state, action):
    sim.reset(options={"init_state": state.copy()})
    obs, _, _, _, _ = sim.step(action)
    return obs.copy()


types = {CRVRobotType, RectangleType, TargetBlockType, TargetSurfaceType}

models = SesameModels(
    observation_space,
    ObjectCentricStateSpace(types),
    action_space,
    transition_fn,
    types,
    predicates,
    lambda obs: obs,  # observations are already object-centric states
    sesame_state_abstractor,
    goal_deriver,
    skills,
)
print(f"Assembled SeSamE models with {len(skills)} skills and {len(predicates)} predicates.")

### Run the planner

`run_sesame` returns a `Plan` whose `actions` are the low-level steps. Two knobs control the search: `max_abstract_plans` (how many symbolic plans to try) and `samples_per_step` (how many parameter samples per skill before giving up). We then execute the returned actions in the environment and animate the result.

The obstruction sits on the target, but our abstraction has no predicate that says a block is *blocking* the goal. So the planner cannot aim to clear it directly — it has to try several abstract plans before it finds one whose refinement happens to move the obstruction out of the way. That is why `max_abstract_plans` is 5 rather than 1. Think about why a single abstract plan is not enough here.

In [ ]:
from IPython.display import display

env = ObjectCentricObstruction2DEnv(num_obstructions=NUM_OBSTRUCTIONS)
obs, _ = env.reset(seed=0)
initial_state = obs.copy()

plan, _ = run_sesame(
    models,
    initial_state,
    seed=0,
    max_abstract_plans=5,
    samples_per_step=2,
    max_skill_horizon=200,
    timeout=60.0,
)

if plan is None:
    print("No plan found -- try increasing max_abstract_plans / samples_per_step, or another seed.")
else:
    print(f"Found a plan with {len(plan.actions)} low-level actions.")

    # Execute the plan, capturing a frame per step.
    frames = [env.render()]
    for action in plan.actions:
        obs, _, _, _, _ = env.step(action)
        frames.append(env.render())

    target = initial_state.get_object_from_name("target_block")
    surface = initial_state.get_object_from_name("target_surface")
    assert is_on(obs, target, surface, {}), "The executed plan did not place the block on the target."
    print("Bilevel planning solved the task!")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis("off")
    image = ax.imshow(frames[0])

    def _draw(i):
        image.set_data(frames[i])
        return [image]

    anim = animation.FuncAnimation(fig, _draw, frames=len(frames), interval=100, blit=True)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

## Exercise: when is one abstract plan enough? ✏️

The instance above needed several abstract plans (`max_abstract_plans=5`). Here is a puzzle that goes the other way.

**Your task:** using `set_state`, construct an initial state in which the obstruction **overlaps the target region**, yet `run_sesame` solves it with `max_abstract_plans=1` — a single, most-direct symbolic plan.

Edit a copy of the reset state with `state.set(obj, feature, value)` — you can move objects and change their sizes — install it with `env.set_state(...)`, and make both checks below pass: the first asserts the obstruction really does overlap the target surface, the second asserts the planner succeeds with just one abstract plan. Think about what has to be true of your state for a single abstract plan to be enough.

In [ ]:
env = ObjectCentricObstruction2DEnv(num_obstructions=1, allow_state_access=True)
obs, _ = env.reset(seed=0)
target = obs.get_objects(TargetBlockType)[0]
surface = obs.get_objects(TargetSurfaceType)[0]
obstruction = [o for o in obs if o.name.startswith("obstruction")][0]


def overlaps_target(state, obj, surf):
    sx, sw = state.get(surf, "x"), state.get(surf, "width")
    ox, ow = state.get(obj, "x"), state.get(obj, "width")
    return ox < sx + sw and ox + ow > sx


# Edit a copy of the reset state, then install it with env.set_state.
custom = obs.copy()
# YOUR CODE HERE
env.set_state(custom)

# Check 1: the obstruction really does overlap the target region.
assert overlaps_target(custom, obstruction, surface), (
    "The obstruction must overlap the target surface's horizontal span."
)

# Check 2: the planner solves it with a single, most-direct abstract plan.
plan, _ = run_sesame(
    models,
    custom.copy(),
    seed=0,
    max_abstract_plans=1,
    samples_per_step=5,
    max_skill_horizon=200,
    timeout=60.0,
)
assert plan is not None, (
    "max_abstract_plans=1 found no plan -- can you reshape the state so the task "
    "is solvable without ever moving the obstruction?"
)
print(
    f"Solved with a single abstract plan in {len(plan.actions)} actions, even "
    "though the obstruction overlaps the target. Why does one plan suffice here?"
)

plt.figure(figsize=(6, 4))
plt.imshow(env.render())
plt.axis("off")
plt.title("Your constructed initial state")
plt.show()

# Where to go next

You've now seen the whole bilevel-planning pipeline on one environment. From here there are two paths — pick whichever appeals to you.

**Path A — explore freely.** Install the benchmark and go play:

```python
%pip install kindergarden
```

Then browse the other environments (`kinder.register_all_environments()`), read the [KinDER website](https://prpl-group.com/kinder-site/) for an overview, and see how the real baselines are built in the [`kinder-baselines`](https://github.com/Princeton-Robot-Planning-and-Learning/kinder-baselines) repo (bilevel planning, the skills/operators model zoo, RL, imitation learning, and more). Try swapping in a different environment and writing models for it.

**Path B — build a pyramid 🔺 (below).** A fresh environment and a real challenge: you write the entire planning domain yourself.

## Path B: build a pyramid 🔺

Here is a brand-new environment, defined from scratch right here — **no targets, surfaces, or obstructions**, just **three identical blocks** on a table. The goal is to arrange them into a pyramid: two blocks side by side as a **base**, and the third resting on top **bridging the seam**.

```
         ┌──────┐
         │ cap  │        <- on top, bridging both
      ┌──┴───┬──┴───┐
      │ base │ base │     <- side by side
 ═════╧══════╧══════╧════  table
```

The catch: the blocks start **spread apart**. So a single "put one on the others" won't work — think about what has to be true *before* the cap can go on, and design predicates and operators that make that happen.

The environment below is given in full. Read it, then build the planning models.

In [ ]:
from dataclasses import dataclass

import numpy as np
from relational_structs import Object, ObjectCentricState, Type
from relational_structs.utils import create_state_from_dict

from kinder.core import FinalConfigMeta
from kinder.envs.kinematic2d.base_env import (
    Kinematic2DRobotEnvConfig,
    ObjectCentricKinematic2DRobotEnv,
)
from kinder.envs.kinematic2d.object_types import (
    CRVRobotType,
    Kinematic2DRobotEnvTypeFeatures,
    RectangleType,
)
from kinder.envs.kinematic2d.structs import SE2Pose, ZOrder
from kinder.envs.kinematic2d.utils import (
    CRVRobotActionSpace,
    create_walls_from_world_boundaries,
)
from kinder.envs.utils import sample_se2_pose, state_2d_has_collision

# A single block type for every movable object (no target/obstruction split).
BlockType = Type("block", parent=RectangleType)
Kinematic2DRobotEnvTypeFeatures[BlockType] = list(
    Kinematic2DRobotEnvTypeFeatures[RectangleType]
)


@dataclass(frozen=True)
class Pyramid2DEnvConfig(Kinematic2DRobotEnvConfig, metaclass=FinalConfigMeta):
    """Config for Pyramid2DEnv()."""

    world_min_x: float = 0.0
    world_max_x: float = (1 + np.sqrt(5)) / 2  # golden ratio
    world_min_y: float = 0.0
    world_max_y: float = 1.0

    min_dx: float = -5e-2
    max_dx: float = 5e-2
    min_dy: float = -5e-2
    max_dy: float = 5e-2
    min_dtheta: float = -np.pi / 16
    max_dtheta: float = np.pi / 16
    min_darm: float = -1e-1
    max_darm: float = 1e-1
    min_vac: float = 0.0
    max_vac: float = 1.0

    robot_base_radius: float = 0.1
    robot_arm_length: float = 2 * robot_base_radius
    robot_gripper_height: float = 0.07
    robot_gripper_width: float = 0.01
    robot_init_pose_bounds: tuple[SE2Pose, SE2Pose] = (
        SE2Pose(world_min_x + robot_base_radius, world_max_y - 2 * robot_base_radius, -np.pi / 2),
        SE2Pose(world_max_x - robot_base_radius, world_max_y - robot_base_radius, -np.pi / 2),
    )

    table_rgb: tuple[float, float, float] = (0.75, 0.75, 0.75)
    table_height: float = 0.1
    table_width: float = world_max_x - world_min_x
    table_pose: SE2Pose = SE2Pose(world_min_x, world_min_y, 0.0)

    # All blocks are identical.
    block_rgb: tuple[float, float, float] = (0.1, 0.3, 0.8)
    block_width: float = 0.15
    block_height: float = 0.09
    num_blocks: int = 3
    # Minimum horizontal gap between blocks in the spread-out start state.
    min_block_separation: float = 0.5 * robot_base_radius

    # Goal tolerances.
    pyramid_adjacency_max_gap: float = 0.05
    pyramid_stack_atol: float = 2e-3

    max_initial_state_sampling_attempts: int = 10_000
    render_dpi: int = 250

In [ ]:
class ObjectCentricPyramid2DEnv(ObjectCentricKinematic2DRobotEnv[Pyramid2DEnvConfig]):
    """Environment where the goal is to stack three blocks into a pyramid."""

    def __init__(self, config: Pyramid2DEnvConfig = Pyramid2DEnvConfig(), **kwargs) -> None:
        super().__init__(config, **kwargs)

    def _block_names(self) -> list[str]:
        return [f"block{i}" for i in range(self.config.num_blocks)]

    def _sample_initial_state(self) -> ObjectCentricState:
        n = self.config.max_initial_state_sampling_attempts
        table_top = self.config.table_pose.y + self.config.table_height
        lo_x = self.config.world_min_x + self.config.robot_base_radius
        hi_x = self.config.world_max_x - self.config.robot_base_radius - self.config.block_width
        for _ in range(n):
            robot_pose = sample_se2_pose(self.config.robot_init_pose_bounds, self.np_random)
            # Spread the blocks out along the table, non-adjacent.
            xs = sorted(self.np_random.uniform(lo_x, hi_x) for _ in range(self.config.num_blocks))
            spread_ok = all(
                xs[i + 1] - (xs[i] + self.config.block_width) >= self.config.min_block_separation
                for i in range(len(xs) - 1)
            )
            if not spread_ok:
                continue
            block_poses = [SE2Pose(x, table_top, 0.0) for x in xs]
            state = self._create_initial_state(robot_pose, block_poses)
            full_state = state.copy()
            full_state.data.update(self.initial_constant_state.data)
            all_objects = set(full_state)
            if state_2d_has_collision(full_state, all_objects, all_objects, {}):
                continue
            if self._blocks_form_pyramid(state):
                continue
            return state
        raise RuntimeError(f"Failed to sample initial state after {n} attempts")

    def _create_constant_initial_state_dict(self) -> dict[Object, dict[str, float]]:
        init_state_dict: dict[Object, dict[str, float]] = {}
        table = Object("table", RectangleType)
        init_state_dict[table] = {
            "x": self.config.table_pose.x,
            "y": self.config.table_pose.y,
            "theta": self.config.table_pose.theta,
            "width": self.config.table_width,
            "height": self.config.table_height,
            "static": True,
            "color_r": self.config.table_rgb[0],
            "color_g": self.config.table_rgb[1],
            "color_b": self.config.table_rgb[2],
            "z_order": ZOrder.ALL.value,
        }
        assert isinstance(self.action_space, CRVRobotActionSpace)
        min_dx, min_dy = self.action_space.low[:2]
        max_dx, max_dy = self.action_space.high[:2]
        wall_state_dict = create_walls_from_world_boundaries(
            self.config.world_min_x,
            self.config.world_max_x,
            self.config.world_min_y,
            self.config.world_max_y,
            min_dx,
            max_dx,
            min_dy,
            max_dy,
        )
        init_state_dict.update(wall_state_dict)
        return init_state_dict

    def _create_initial_state(
        self, robot_pose: SE2Pose, block_poses: list[SE2Pose]
    ) -> ObjectCentricState:
        init_state_dict: dict[Object, dict[str, float]] = {}
        robot = Object("robot", CRVRobotType)
        init_state_dict[robot] = {
            "x": robot_pose.x,
            "y": robot_pose.y,
            "theta": robot_pose.theta,
            "base_radius": self.config.robot_base_radius,
            "arm_joint": self.config.robot_base_radius,
            "arm_length": self.config.robot_arm_length,
            "vacuum": 0.0,
            "gripper_height": self.config.robot_gripper_height,
            "gripper_width": self.config.robot_gripper_width,
        }
        for name, pose in zip(self._block_names(), block_poses):
            block = Object(name, BlockType)
            init_state_dict[block] = {
                "x": pose.x,
                "y": pose.y,
                "theta": pose.theta,
                "width": self.config.block_width,
                "height": self.config.block_height,
                "static": False,
                "color_r": self.config.block_rgb[0],
                "color_g": self.config.block_rgb[1],
                "color_b": self.config.block_rgb[2],
                "z_order": ZOrder.ALL.value,
            }
        return create_state_from_dict(init_state_dict, Kinematic2DRobotEnvTypeFeatures)

    def _blocks_form_pyramid(self, state: ObjectCentricState) -> bool:
        blocks = state.get_objects(BlockType)
        if len(blocks) != 3:
            return False
        table_top = self.config.table_pose.y + self.config.table_height
        for cap in blocks:
            base = [b for b in blocks if b is not cap]
            left, right = sorted(base, key=lambda o: state.get(o, "x"))
            left_lo = state.get(left, "x")
            left_hi = left_lo + state.get(left, "width")
            right_lo = state.get(right, "x")
            right_hi = right_lo + state.get(right, "width")
            # Base blocks rest on the table.
            if abs(state.get(left, "y") - table_top) > self.config.pyramid_stack_atol:
                continue
            if abs(state.get(right, "y") - table_top) > self.config.pyramid_stack_atol:
                continue
            # Base blocks are adjacent (a small gap is allowed).
            gap = right_lo - left_hi
            if not (-1e-3 <= gap <= self.config.pyramid_adjacency_max_gap):
                continue
            # Cap rests on top of the base.
            base_top = state.get(left, "y") + state.get(left, "height")
            if abs(state.get(cap, "y") - base_top) > self.config.pyramid_stack_atol:
                continue
            # Cap bridges the seam (a corner over each base block).
            cap_lo = state.get(cap, "x")
            cap_hi = cap_lo + state.get(cap, "width")
            if left_lo <= cap_lo <= left_hi and right_lo <= cap_hi <= right_hi:
                return True
        return False

    def _get_reward_and_done(self) -> tuple[float, bool]:
        assert self._current_state is not None
        terminated = self._blocks_form_pyramid(self._current_state)
        return -1.0, terminated

### The environment

Three blocks scattered on a table. Run it and look.

In [ ]:
pyramid_env = ObjectCentricPyramid2DEnv()
obs, _ = pyramid_env.reset(seed=0)
print("objects:", sorted(o.name for o in obs))

plt.figure(figsize=(6, 4))
plt.imshow(pyramid_env.render())
plt.axis("off")
plt.title("Pyramid2D initial state (seed=0)")
plt.show()

### Build the planning domain (your turn ✏️)

This is the capstone: **you** design the whole domain. The base waypoint controller and the planner plumbing are provided; everything symbolic is yours to write. You'll fill in, in order:

1. the **predicates**,
2. the **state abstractor** (their classifiers),
3. the **operators**,
4. the **goal**.

Reuse the patterns from the obstruction walkthrough above. The new idea is that the base blocks start apart, so you need a predicate that captures *two blocks side by side* and an operator that brings them there before the cap goes on. The constants and lifted variables below are given so the provided controllers and the final assembly line up with your definitions.

In [ ]:
from gymnasium.spaces import Box
from relational_structs.spaces import ObjectCentricStateSpace

# Geometry constants (given).
PADDING = 1e-4       # grasp/lift clearance
CLEARANCE = 1e-4     # rest a hair above a support so the contact is not a collision
ADJ_GAP = Pyramid2DEnvConfig().pyramid_adjacency_max_gap
TABLE_TOP = Pyramid2DEnvConfig().table_pose.y + Pyramid2DEnvConfig().table_height

# Lifted variables for the operators and controllers (given).
robot_var = Variable("?robot", CRVRobotType)
block_var = Variable("?block", BlockType)
held_var = Variable("?held", BlockType)
base_var = Variable("?base", BlockType)
cap_var = Variable("?cap", BlockType)
b1_var = Variable("?b1", BlockType)
b2_var = Variable("?b2", BlockType)

In [ ]:
# TODO: define the predicates and collect them in `predicates`.
# YOUR CODE HERE

In [ ]:
# TODO: implement state_abstractor(state) -> RelationalAbstractState.
# YOUR CODE HERE

In [ ]:
# TODO: define the lifted operators the pyramid needs.
# YOUR CODE HERE

### The controllers (your turn ✏️)

Each operator needs a controller that turns it into low-level actions. Subclass the `Kinematic2dRobotController` base from the obstruction section and implement three, each with `sample_parameters(state, rng)`, `_generate_waypoints(state)` (rise to a safe height, move across, descend), and `_get_vacuum_actions()`:

- **`PyramidPickController`** `[robot, block]` — grasp a block off the table (vacuum off→on);
- **`PyramidMakeAdjacentController`** `[robot, held, base]` — place the held block beside `base` on the table (vacuum on→off);
- **`PyramidPlaceCapController`** `[robot, cap, b1, b2]` — lower the held cap onto the two adjacent base blocks, centred so a bottom corner lands on each.

Geometry tip for the placers: rest a block a hair above its support and compute the descent target from the block's **measured** position (it tracks the robot), so block-on-block contact lands cleanly instead of registering as a collision.

In [ ]:
# TODO: implement the three pyramid controllers (see the prompt above).
# YOUR CODE HERE

### The goal (your turn ✏️)

Return the goal that describes a finished pyramid (the cap on top of the two base blocks). Pick one block as the cap by convention — the blocks are identical.

In [ ]:
# TODO: implement goal_deriver(state) -> RelationalAbstractGoal.
# YOUR CODE HERE

### Assemble and run

Pair each operator with its controller, build the `SesameModels`, and plan. The base blocks start apart, so — as in the obstruction finale — SeSamE needs a few abstract plans to find one it can refine; this takes a few seconds.

In [ ]:
# TODO: pair each operator with its controller into skills, write the
# transition function, and build `models = SesameModels(...)`.
# YOUR CODE HERE

### Watch it build the pyramid

In [ ]:
from IPython.display import display

env = ObjectCentricPyramid2DEnv()
obs, _ = env.reset(seed=0)
initial_state = obs.copy()

plan, _ = run_sesame(
    models, initial_state, seed=0,
    max_abstract_plans=5, samples_per_step=10, max_skill_horizon=300, timeout=60.0,
)

if plan is None:
    print("No plan found -- try more abstract plans / samples, or another seed.")
else:
    print(f"Found a plan with {len(plan.actions)} low-level actions.")
    frames = [env.render()]
    for action in plan.actions:
        obs, _, _, _, _ = env.step(action)
        frames.append(env.render())
    assert env._blocks_form_pyramid(obs), "The executed plan did not build a pyramid."
    print("Pyramid built!")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis("off")
    image = ax.imshow(frames[0])

    def _draw(i):
        image.set_data(frames[i])
        return [image]

    anim = animation.FuncAnimation(fig, _draw, frames=len(frames), interval=100, blit=True)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))